# 🚀 Two-Agent News Sentiment Analyzer with Agentic Tools

This notebook implements a dynamic RAG pipeline for financial sentiment analysis using a two-agent FinRobot workflow with tool calling:
1. Instantiates a **Sentiment Scorer Agent** (loaded from `sentiment_prompt_with_tools.txt`) equipped with the custom tool `prepare_articles_tool`.
2. The Scorer Agent calls `prepare_articles_tool` which fetches the latest news, queries the FAISS vector database for calibration examples, and structures the context inline.
3. The Scorer Agent scores the articles using these calibration benchmarks.
4. Instantiates a **Senior Sentiment Analyst (CIO) Agent** (loaded from `cio_prompt.txt`) to review, average, and compile the final JSON report.

In [ ]:
import sys
import os
from dotenv import load_dotenv

# Ensure the current directory is in the python path for importing modules
notebook_dir = os.getcwd()
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

project_root = os.path.dirname(notebook_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

sentiment_dir = os.path.join(project_root, "sentiment")
if sentiment_dir not in sys.path:
    sys.path.insert(0, sentiment_dir)

# Load environment variables from .env.local
load_dotenv("../sentiment/.env.local")

In [ ]:
import json
import datetime
import pandas as pd
import autogen
from finrobot.agents.workflow import FinRobot
from autogen import UserProxyAgent
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Import refactored utility functions
from sentiment.functions.utils.read_and_clean import read_file_content, extract_and_clean_response
from sentiment.functions.utils.build import build_vector_store
from sentiment.functions.utils.config import generate_config

# Import tools and tool registration helpers
from sentiment.functions.tools.prepare_articles import prepare_articles_tool, set_db_instance

# Read and clean environment variables
nvidia_embedding_model = os.getenv("NVIDIA_EMBEDDING_MODEL", "nvidia/nv-embed-v1").strip('"\' ')
nvidia_base_model = os.getenv("NVIDIA_BASE_MODEL", "").strip('"\' ')
nvidia_api_endpoint = os.getenv("NVIDIA_API_ENDPOINT", "https://integrate.api.nvidia.com/v1").strip('"\' ')
nvidia_api_key = os.getenv("NVIDIA_API_KEY", "").strip('"\' ')

print(f"Initializing NVIDIA Embeddings wrapper ({nvidia_embedding_model})...")
embeddings = NVIDIAEmbeddings(
    model=nvidia_embedding_model,
    nvidia_api_key=nvidia_api_key,
    base_url=nvidia_api_endpoint
)

In [ ]:
def create_scorer_agent(prompt_path, schema_path, llm_config):
    """Instantiates and returns the Sentiment Scorer agent equipped with prepare_articles_tool."""
    schema_str = read_file_content(schema_path)
    scorer_prompt_template = read_file_content(prompt_path)
    
    scorer_profile = scorer_prompt_template.format(
        SCHEMA=schema_str,
        EXAMPLES="Use the matching 'calibration_examples' list provided inline inside the user message for each article."
    )
    
    return FinRobot(
        agent_config={
            "name": "Sentiment_Scorer",
            "description": "Scoration specialist agent equipped with article fetching and calibration tool.",
            "profile": scorer_profile,
            "toolkits": [prepare_articles_tool]
        },
        llm_config=llm_config
    )

def create_cio_agent(prompt_path, schema_path, output_schema_path, scored_articles_path, llm_config):
    """Instantiates and returns the Senior Sentiment Analyst (CIO) agent."""
    cio_prompt_template = read_file_content(prompt_path)
    schema_str = read_file_content(schema_path)
    output_str = read_file_content(output_schema_path)
    scored_articles_str = read_file_content(scored_articles_path)
    
    cio_profile = cio_prompt_template.format(
        SCHEMA=schema_str,
        EXAMPLES=scored_articles_str,
        OUTPUT=output_str
    )
    
    return FinRobot(
        agent_config={
            "name": "Senior_Sentiment_Analyst",
            "description": "Consolidation and aggregation agent.",
            "profile": cio_profile,
            "toolkits": []
        },
        llm_config=llm_config
    )

In [ ]:
hf_api_key = os.getenv("HUGGINGFACE_API_KEY", "").strip('"\' ')
hf_model_name = os.getenv("HUGGINGFACE_MODEL_NAME_FEATHERLESS", "curiousily/Llama-3-8B-Instruct-Finance-RAG").strip('"\' ')
hf_base_url = os.getenv("HUGGINGFACE_BASE_URL", "https://router.huggingface.co/v1").strip('"\' ')

print(f"HF Model Name: {hf_model_name}")
print(f"HF Base URL: {hf_base_url}")
print(f"HF API Key exists: {bool(hf_api_key)}")

config_list = generate_config(hf_model_name, hf_base_url, hf_api_key)
base_config_list = generate_config(nvidia_base_model, nvidia_api_endpoint, nvidia_api_key)

llm_config = {"config_list": config_list, "model": hf_model_name}
base_llm_config = {"config_list": base_config_list, "model": nvidia_base_model}

In [ ]:
ticker = "AAPL"
news_limit = 5  # Score top 5 articles

In [ ]:
# Build vector store
db = build_vector_store("../sentiment/data/financial_sentiment.csv", embeddings, limit_rows=300)

# Register database instance with the tool module
set_db_instance(db)

# Instantiate the UserProxyAgent
user_proxy = UserProxyAgent(
    name="User_Proxy",
    human_input_mode="NEVER",
    is_termination_msg=lambda x: x.get("content", "") and "TERMINATE" in x.get("content", ""),
    max_consecutive_auto_reply=5,
    code_execution_config={"use_docker": False}
)

In [ ]:
# Instantiate scorer and CIO agents
scorer_agent = create_scorer_agent(
    prompt_path="../sentiment/prompts/sentiment_prompt_with_tools.txt",
    schema_path="../sentiment/schema_json/scorer_schema.json",
    llm_config=llm_config
)

cio_agent = create_cio_agent(
    prompt_path="../sentiment/prompts/cio_prompt.txt",
    schema_path="../sentiment/schema_json/sentiment_schema.json",
    output_schema_path="../sentiment/schema_json/cio_output_schema.json",
    scored_articles_path="../sentiment/schema_json/cio_scored_articles.json",
    llm_config=base_llm_config
)

In [ ]:
# Step 1: Sentiment Scorer fetching and analyzing articles via prepare_articles_tool
print(f"Initiating Scorer agent for ticker '{ticker}' (limit {news_limit})...")
scorer_task = (
    f"Fetch and analyze the latest news articles for ticker '{ticker}' with a limit of {news_limit}. "
    "Score the articles according to your instructions and respond with the list of scored articles."
)

user_proxy.initiate_chat(scorer_agent, message=scorer_task)
scorer_msg = extract_and_clean_response(user_proxy, scorer_agent, is_json=False)

print("\n--- Scorer Output ---")
print(scorer_msg)

In [ ]:
# Step 2: Senior Sentiment Analyst consolidating scores
print(f"Initiating Senior Sentiment Analyst agent to consolidate scored articles...")
cio_task = (
    "Please consolidate the following scored articles into the final JSON report according to your instructions:\n\n"
    f"{scorer_msg}\n\n"
    "Generate the final JSON block matching the schema."
)

user_proxy.initiate_chat(cio_agent, message=cio_task)
final_report_msg = extract_and_clean_response(user_proxy, cio_agent, is_json=True)

print("\n================ FINAL REPORT ================")
try:
    final_report = json.loads(final_report_msg)
    print(json.dumps(final_report, indent=2))
except Exception as e:
    print(f"Error parsing final report: {e}")
    print("Raw Output:")
    print(final_report_msg)

In [ ]:
print(final_report_msg)